# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdubakr77/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from getpass import getpass
import duckdb
from huggingface_hub import hf_hub_download
import os

# List available files in the dataset to see its structure
from huggingface_hub import HfApi


hf_token = getpass("Enter your Hugging Face token: ")
print("Token loaded:", hf_token is not None and len(hf_token) > 0)

os.environ["HF_TOKEN"] = hf_token
con = duckdb.connect()
con.execute("SET enable_object_cache=true")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}')")

Token loaded: True


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one client's content page on one calendar day (identified by `client_hash_id` + `content_hash_id` + `report_date`).

**Time window:** the full warehouse spans January 2025 through June 2026 (18 monthly partitions). I am working with the March 2026 partition as my mid-panel month for verification, as instructed.

**Verified below:** on the March 2026 partition, row count (9,841,378) exactly matches the count of unique (client, content, date) combinations, confirming the grain holds with zero duplicate rows. The date range for this partition runs from 2026-03-01 to 2026-03-31, confirming it is a single calendar month as expected.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=hf_token)
for f in files:
    print(f)

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token loaded: True
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_dai

In [ ]:
local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

print("Downloaded to:", local_path)

df_sample = con.execute(f"""
    SELECT * FROM read_parquet('{local_path}')
    LIMIT 5
""").df()

print(df_sample.shape)
df_sample

Downloaded to: C:\Users\ibrah.HIMA\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-03\data_0.parquet
(5, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [21]:
grain_check = con.execute(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_combinations
    FROM read_parquet('{local_path}')
""").df()

grain_check

,total_rows,unique_combinations
0,9841378,9841378


In [22]:
date_range = con.execute(f"""
    SELECT MIN(report_date) AS earliest_date, MAX(report_date) AS latest_date
    FROM read_parquet('{local_path}')
""").df()

date_range

,earliest_date,latest_date
0,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (inputs the model would use):**
`gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`

**Label (what the model predicts):**
No pre-built label column exists in this table. The label would need to be derived by comparing a page's performance (for example `gsc_impressions` or `ga4_sessions`) across two time windows to define a trend direction, similar to the `trend_direction` proxy I used in the smaller starter dataset.

**Context (useful, but not model inputs):**
`report_date`, `client_hash_id`, `content_hash_id`, `month`

**Excluded (deliberately not used, with reason):**
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`: these are availability flags, not performance signals. They matter for filtering rows before analysis, not as model features.
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`, `scroll_events`: excluded for this lane, not because they are missing (verified: about 69% of rows have non-null values for `ai_chatgpt` and `scroll_events`), but because they measure a different signal (AI-referral traffic and on-page engagement) than the refresh-priority question this lane is built around. Mixing them in now would blur what the model is actually scoring.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
all_columns = con.execute(f"""
    SELECT * FROM read_parquet('{local_path}') LIMIT 0
""").df().columns.tolist()

print(all_columns)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [24]:
null_check = con.execute(f"""
    SELECT 
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ai_chatgpt IS NOT NULL THEN 1 ELSE 0 END) AS ai_chatgpt_non_null,
        SUM(CASE WHEN scroll_events IS NOT NULL THEN 1 ELSE 0 END) AS scroll_events_non_null
    FROM read_parquet('{local_path}')
""").df()

null_check

,total_rows,ai_chatgpt_non_null,scroll_events_non_null
0,9841378,6822637.0,6822637.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim made in sections 1 and 2 is backed by a query below.

**Query 4 (Availability flag vs actual data, new check):** grouped rows by `gsc_data_available` and checked whether `gsc_impressions` is actually populated in each group. Surprising result: `gsc_impressions` is 100% non-null in both groups (3,611,061 rows where the flag is True, and 6,230,317 rows where it is False). This means `gsc_data_available` does not reliably indicate whether the impressions column itself is populated, it likely reflects a different eligibility rule (for example, whether GSC is connected for that client at all) rather than row-level data presence. Filtering with `IS TRUE` on this flag would silently exclude 6.2 million rows that actually have usable data, this is exactly the kind of assumption the contract format is meant to catch.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
availability_check = con.execute(f"""
    SELECT 
        gsc_data_available,
        COUNT(*) AS row_count,
        SUM(CASE WHEN gsc_impressions IS NOT NULL THEN 1 ELSE 0 END) AS impressions_present
    FROM read_parquet('{local_path}')
    GROUP BY gsc_data_available
""").df()

availability_check

,gsc_data_available,row_count,impressions_present
0,False,6230317,6230317.0
1,True,3611061,3611061.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Single-month view:** everything verified above is based on the March 2026 partition only. Patterns confirmed here (grain, the `gsc_data_available` flag issue) are not yet confirmed to hold across all 18 months; seasonal or early-data issues in other months have not been checked.

**The `gsc_data_available` flag cannot be trusted as a row-level filter.** As shown in section 3, it does not reliably indicate whether `gsc_impressions` is populated, it appears to reflect a client-level eligibility rule instead. Any query relying on `IS TRUE` for this flag risks silently dropping millions of usable rows, this needs to be re-verified against the actual column values, not the flag, in every future query.

**AI-referral and engagement columns (`ai_chatgpt`, `scroll_events`, etc.) are only about 69% populated** in this partition. This data was assumed to not be tracked yet based on an early, small sample, that assumption was wrong. However, the 31% of missing rows still need investigation: it is not yet known whether they are missing because of a rollout timing issue, a client-level tracking gap, or something else.

**No GA4-only or GSC-only asymmetry has been checked yet.** The `client_has_gsc` and `client_has_ga4` flags suggest some clients only report one data source, this partition has not been broken down by that split, so it is unknown how much of the row count comes from GSC-only clients versus GA4-only clients versus both.

**This data cannot establish causation.** Even with a clean label and verified features, any model built on this data can only show correlation between signals like staleness or visibility and declining performance, it cannot prove that refreshing a page will cause traffic recovery.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.